In [ ]:
import sys
import os
sys.path.insert(0, '..')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import load_and_clean, build_pipeline, NUMERIC_COLS, CATEGORICAL_COLS
from src.features import add_features

%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Phase 1: Exploratory Data Analysis

In [ ]:
df_raw = pd.read_csv('../data/telco_churn.csv')
print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
churn_rate = df_raw['Churn'].value_counts(normalize=True)
print(churn_rate)

fig, ax = plt.subplots(figsize=(6, 4))
churn_rate.plot(kind='bar', ax=ax, color=['steelblue', 'salmon'])
ax.set_title('Overall Churn Rate')
ax.set_xlabel('Churn')
ax.set_ylabel('Proportion')
ax.set_xticklabels(['No Churn', 'Churn'], rotation=0)
for i, v in enumerate(churn_rate):
    ax.text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("Rows with blank TotalCharges:")
print(df_raw[df_raw['TotalCharges'].str.strip() == ''][['customerID', 'tenure', 'TotalCharges']])

In [ ]:
contract_churn = df_raw.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean()
).reset_index()
contract_churn.columns = ['Contract', 'ChurnRate']

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(contract_churn['Contract'], contract_churn['ChurnRate'], color=['salmon', 'steelblue', 'seagreen'])
ax.set_title('Churn Rate by Contract Type')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate')
for i, row in contract_churn.iterrows():
    ax.text(i, row['ChurnRate'] + 0.005, f"{row['ChurnRate']:.1%}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
df_raw['tenure_bucket'] = pd.cut(
    df_raw['tenure'],
    bins=[0, 12, 24, 48, float('inf')],
    labels=['0-12', '13-24', '25-48', '49+'],
    include_lowest=True
)
tenure_churn = df_raw.groupby('tenure_bucket')['Churn'].apply(
    lambda x: (x == 'Yes').mean()
).reset_index()
tenure_churn.columns = ['tenure_bucket', 'ChurnRate']

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(tenure_churn['tenure_bucket'].astype(str), tenure_churn['ChurnRate'],
       color=['salmon', 'orange', 'steelblue', 'seagreen'])
ax.set_title('Churn Rate by Tenure Bucket')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Churn Rate')
for i, row in tenure_churn.iterrows():
    ax.text(i, row['ChurnRate'] + 0.005, f"{row['ChurnRate']:.1%}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
internet_churn = df_raw.groupby('InternetService')['Churn'].apply(
    lambda x: (x == 'Yes').mean()
).reset_index()
internet_churn.columns = ['InternetService', 'ChurnRate']

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(internet_churn['InternetService'], internet_churn['ChurnRate'],
       color=['steelblue', 'salmon', 'seagreen'])
ax.set_title('Churn Rate by Internet Service Type')
ax.set_xlabel('Internet Service')
ax.set_ylabel('Churn Rate')
for i, row in internet_churn.iterrows():
    ax.text(i, row['ChurnRate'] + 0.005, f"{row['ChurnRate']:.1%}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
corr_data = df_raw[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
corr_data['TotalCharges'] = pd.to_numeric(corr_data['TotalCharges'], errors='coerce')
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()

**EDA Checkpoint:** Customers most likely to churn: month-to-month contracts, tenure < 12 months, high MonthlyCharges, Fiber optic internet, no tech support.

# Phase 2 & 3: Cleaning, Preprocessing & Feature Engineering

In [ ]:
# Use src/ functions — no inline duplication
df = load_and_clean('../data/telco_churn.csv')
df = add_features(df)
print(df.shape)
df.head()

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Churn rate train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

In [ ]:
# Feature correlation check for new features
df_corr = df[['num_services', 'avg_monthly_spend', 'has_streaming', 'Churn']].corr()
sns.heatmap(df_corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('New Feature Correlations with Churn')
plt.show()

# Phase 4: Handling Class Imbalance

In [ ]:
print(f"Class distribution: {y_train.value_counts().to_dict()}")
print(f"Churn rate: {y_train.mean():.3f}")
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

In [ ]:
# Build a simple pipeline for SMOTE comparison (uses default XGB in build_pipeline)
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Fit preprocessor to get encoded data for SMOTE
preprocessor_only = build_pipeline().named_steps['preprocessor']
preprocessor_only.fit(X_train)
X_train_enc = preprocessor_only.transform(X_train)
X_test_enc = preprocessor_only.transform(X_test)

# SMOTE on training only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_enc, y_train)

xgb_smote = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
xgb_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = xgb_smote.predict(X_test_enc)
print("SMOTE XGBoost:\n", classification_report(y_test, y_pred_smote))

**Decision:** Using class weights (scale_pos_weight) in the deployed model — simpler, no risk of SMOTE leakage at inference time. Performance is comparable on this dataset.

# Phase 5: Modeling — Baseline to XGBoost

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_pipeline = build_pipeline(model=lr)
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]
print("Logistic Regression:")
print(classification_report(y_test, y_pred_lr))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42)
rf_pipeline = build_pipeline(model=rf)
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]
print("Random Forest:")
print(classification_report(y_test, y_pred_rf))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

In [ ]:
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)
xgb_pipeline = build_pipeline(model=xgb_model)
xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test)
y_prob_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]
print("XGBoost:")
print(classification_report(y_test, y_pred_xgb))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_xgb):.4f}")

# Phase 6: Hyperparameter Tuning (XGBoost)

In [ ]:
param_grid = {
    'model__max_depth': [3, 5, 7],
    'model__n_estimators': [100, 300, 500],
    'model__learning_rate': [0.01, 0.05, 0.1],
}

xgb_base = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)
xgb_pipeline_tune = build_pipeline(model=xgb_base)

grid = GridSearchCV(
    xgb_pipeline_tune,
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)
print(f"Best params: {grid.best_params_}")
print(f"Best CV F1: {grid.best_score_:.4f}")

best_pipeline = grid.best_estimator_
y_pred_tuned = best_pipeline.predict(X_test)
y_prob_tuned = best_pipeline.predict_proba(X_test)[:, 1]
print("\nTuned XGBoost:")
print(classification_report(y_test, y_pred_tuned))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_tuned):.4f}")

# Phase 7: Evaluation Deep-Dive

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_tuned, ax=ax, colorbar=False)
ax.set_title('Confusion Matrix — Tuned XGBoost (threshold=0.5)')
plt.tight_layout()
plt.savefig('../notebooks/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, y_prob_tuned, ax=ax, name='Tuned XGBoost')
ax.set_title('ROC Curve — Tuned XGBoost')
plt.tight_layout()
plt.savefig('../notebooks/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("Threshold Analysis:")
print("-" * 60)
for thresh in [0.3, 0.35, 0.5]:
    y_thresh = (y_prob_tuned >= thresh).astype(int)
    print(f"\nThreshold = {thresh}")
    print(classification_report(y_test, y_thresh, target_names=['No Churn', 'Churn']))

**Business Decision:** We lower the threshold to 0.35. Since a missed churner (false negative) costs more than a wasted retention offer (false positive), we accept lower precision in exchange for higher recall on the churn class.

# Phase 8: SHAP Interpretability

In [ ]:
xgb_model_fit = best_pipeline.named_steps['model']
feature_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out()
X_test_transformed = best_pipeline.named_steps['preprocessor'].transform(X_test)

explainer = shap.TreeExplainer(xgb_model_fit)
shap_values = explainer.shap_values(X_test_transformed)

# Global summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig('../notebooks/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Individual prediction — first churner in test set
churner_idx = y_test[y_test == 1].index[0]
pos = list(X_test.index).index(churner_idx)
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[pos],
        base_values=explainer.expected_value,
        feature_names=list(feature_names)
    )
)

## Business Insight

Month-to-month contract, tenure < 12 months, and absence of tech support are the three strongest churn signals. High MonthlyCharges amplifies risk, while longer tenure and two-year contracts are strong retention indicators. Customers who bundle more services (higher num_services) show lower churn probability, suggesting that cross-selling reduces attrition risk.

# Save Final Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(best_pipeline, '../models/churn_pipeline.pkl')
print("Best pipeline saved to ../models/churn_pipeline.pkl")